In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('datathon-2026-round-1/'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/datathon-2026-round-1/products.csv
/kaggle/input/competitions/datathon-2026-round-1/sample_submission.csv
/kaggle/input/competitions/datathon-2026-round-1/promotions.csv
/kaggle/input/competitions/datathon-2026-round-1/shipments.csv
/kaggle/input/competitions/datathon-2026-round-1/order_items.csv
/kaggle/input/competitions/datathon-2026-round-1/reviews.csv
/kaggle/input/competitions/datathon-2026-round-1/inventory.csv
/kaggle/input/competitions/datathon-2026-round-1/returns.csv
/kaggle/input/competitions/datathon-2026-round-1/sales.csv
/kaggle/input/competitions/datathon-2026-round-1/orders.csv
/kaggle/input/competitions/datathon-2026-round-1/geography.csv
/kaggle/input/competitions/datathon-2026-round-1/customers.csv
/kaggle/input/competitions/datathon-2026-round-1/baseline.ipynb
/kaggle/input/competitions/datathon-2026-round-1/payments.csv
/kaggle/input/competitions/datathon-2026-round-1/web_traffic.csv


In [2]:
import pandas as pd
import os

# Đường dẫn mặc định của cuộc thi trên Kaggle
BASE_PATH = 'datathon-2026-round-1/'

# Từ điển cấu hình các cột ngày tháng cho từng file
DATE_CONFIG = {
    'sales.csv': ['Date'],
    'web_traffic.csv': ['date'],
    'orders.csv': ['order_date'],
    'promotions.csv': ['start_date', 'end_date'],
    'customers.csv': ['signup_date'],
    'shipments.csv': ['ship_date', 'delivery_date'],
    'returns.csv': ['return_date'],
    'reviews.csv': ['review_date'],
    'inventory.csv': ['snapshot_date']
}

def load_all_data(base_path):
    data = {}
    # Lấy danh sách tất cả file .csv trong thư mục
    files = [f for f in os.listdir(base_path) if f.endswith('.csv')]
    
    print(f"🚀 Đang bắt đầu load {len(files)} file dữ liệu...")
    
    for file in files:
        file_path = os.path.join(base_path, file)
        df_name = file.split('.')[0] # Lấy tên file bỏ đuôi .csv
        
        # Kiểm tra xem file có cột ngày tháng cần parse không
        parse_dates = DATE_CONFIG.get(file, False)
        
        try:
            data[df_name] = pd.read_csv(file_path, parse_dates=parse_dates)
            print(f"✅ Loaded: {df_name:<20} | Shape: {str(data[df_name].shape):<15}")
        except Exception as e:
            print(f"❌ Lỗi khi load {file}: {e}")
            
    return data

# Thực thi
dfs = load_all_data(BASE_PATH)

# --- TRUY XUẤT NHANH ---
# Ví dụ: dfs['sales'], dfs['products'], dfs['web_traffic']

🚀 Đang bắt đầu load 14 file dữ liệu...
✅ Loaded: products             | Shape: (2412, 8)      
✅ Loaded: sample_submission    | Shape: (548, 3)       
✅ Loaded: promotions           | Shape: (50, 10)       
✅ Loaded: shipments            | Shape: (566067, 4)    


/tmp/ipykernel_55/925050949.py:35: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  data[df_name] = pd.read_csv(file_path, parse_dates=parse_dates)


✅ Loaded: order_items          | Shape: (714669, 7)    
✅ Loaded: reviews              | Shape: (113551, 7)    
✅ Loaded: inventory            | Shape: (60247, 17)    
✅ Loaded: returns              | Shape: (39939, 7)     
✅ Loaded: sales                | Shape: (3833, 3)      
✅ Loaded: orders               | Shape: (646945, 8)    
✅ Loaded: geography            | Shape: (39948, 4)     
✅ Loaded: customers            | Shape: (121930, 7)    
✅ Loaded: payments             | Shape: (646945, 4)    
✅ Loaded: web_traffic          | Shape: (3652, 7)      


In [4]:
# Trích xuất dataframe từ dictionary dfs bạn đã tạo
orders = dfs['orders'].copy()

# Bước 1: Sắp xếp theo khách hàng và thời gian (Bắt buộc để diff đúng)
orders = orders.sort_values(by=['customer_id', 'order_date'])

# Bước 2: Tính số ngày chênh lệch giữa các đơn hàng liên tiếp của từng khách hàng
# groupby('customer_id') giúp việc tính toán không bị nhảy từ khách hàng này sang khách hàng khác
orders['gap'] = orders.groupby('customer_id')['order_date'].diff().dt.days

# Bước 3: Loại bỏ các giá trị rỗng (NaN)
# Các giá trị NaN xuất hiện ở đơn hàng đầu tiên của mỗi khách hàng
inter_order_gaps = orders['gap'].dropna()

# Bước 4: Tính trung vị
median_gap = inter_order_gaps.median()

print(f"Giá trị trung vị tính được: {median_gap} ngày")

Giá trị trung vị tính được: 144.0 ngày


In [6]:
# 1. Trích xuất bảng products
products = dfs['products'].copy()

# 2. Áp dụng công thức tính tỷ suất lợi nhuận cho từng dòng sản phẩm
# Margin = (Price - COGS) / Price
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']

# 3. Tính trung bình tỷ suất lợi nhuận theo từng phân khúc (segment)
segment_margin = products.groupby('segment')['gross_margin'].mean()

# 4. Tìm phân khúc có giá trị cao nhất
best_segment = segment_margin.idxmax()
highest_value = segment_margin.max()

print("--- KẾT QUẢ Q2 ---")
print(segment_margin.sort_values(ascending=False))
print(f"\n=> Phân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất là: {best_segment} ({highest_value:.2%})")

--- KẾT QUẢ Q2 ---
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64

=> Phân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất là: Standard (31.34%)


In [7]:
# 1. Trích xuất bảng returns và products
returns_df = dfs['returns'].copy()
products_df = dfs['products'].copy()

# 2. Thực hiện join (nối) hai bảng dựa trên cột chung là product_id
# Chúng ta dùng 'inner join' để chỉ lấy những bản ghi trả hàng có thông tin sản phẩm tương ứng
merged_returns = pd.merge(returns_df, products_df, on='product_id', how='inner')

# 3. Lọc các sản phẩm thuộc danh mục 'Streetwear'
streetwear_returns = merged_returns[merged_returns['category'] == 'Streetwear']

# 4. Đếm số lần xuất hiện của từng lý do trả hàng (return_reason)
reason_counts = streetwear_returns['return_reason'].value_counts()

# 5. Tìm lý do phổ biến nhất
most_common_reason = reason_counts.idxmax()
count_value = reason_counts.max()

print("--- KẾT QUẢ Q3 ---")
print("Thống kê các lý do trả hàng trong danh mục Streetwear:")
print(reason_counts)
print(f"\n=> Lý do trả hàng xuất hiện nhiều nhất là: {most_common_reason} (với {count_value} lượt)")

--- KẾT QUẢ Q3 ---
Thống kê các lý do trả hàng trong danh mục Streetwear:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

=> Lý do trả hàng xuất hiện nhiều nhất là: wrong_size (với 7626 lượt)


In [8]:
# 1. Trích xuất bảng web_traffic
web_traffic = dfs['web_traffic'].copy()

# 2. Nhóm theo nguồn truy cập (traffic_source) và tính trung bình cộng của bounce_rate
# Chúng ta sử dụng hàm .mean() để tính giá trị trung bình trên tất cả các ngày
source_bounce_stats = web_traffic.groupby('traffic_source')['bounce_rate'].mean()

# 3. Tìm nguồn có tỷ lệ thoát thấp nhất (min)
best_traffic_source = source_bounce_stats.idxmin()
lowest_bounce_rate = source_bounce_stats.min()

print("--- KẾT QUẢ Q4 ---")
print("Thống kê tỷ lệ thoát trung bình theo nguồn truy cập:")
print(source_bounce_stats.sort_values()) # Sắp xếp từ thấp đến cao để dễ quan sát
print(f"\n=> Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất là: {best_traffic_source} ({lowest_bounce_rate:.2%})")

--- KẾT QUẢ Q4 ---
Thống kê tỷ lệ thoát trung bình theo nguồn truy cập:
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64

=> Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất là: email_campaign (0.45%)


In [9]:
# 1. Trích xuất bảng order_items
order_items = dfs['order_items'].copy()

# 2. Đếm số dòng có áp dụng khuyến mãi (promo_id không phải là NaN/null)
promo_count = order_items['promo_id'].notnull().sum()

# 3. Tính tổng số dòng trong bảng
total_count = len(order_items)

# 4. Tính tỷ lệ phần trăm
promo_percentage = (promo_count / total_count) * 100

print("--- KẾT QUẢ Q5 ---")
print(f"Số dòng có khuyến mãi: {promo_count}")
print(f"Tổng số dòng: {total_count}")
print(f"\n=> Tỷ lệ phần trăm có áp dụng khuyến mãi là: {promo_percentage:.2f}%")

--- KẾT QUẢ Q5 ---
Số dòng có khuyến mãi: 276316
Tổng số dòng: 714669

=> Tỷ lệ phần trăm có áp dụng khuyến mãi là: 38.66%


In [10]:
# 1. Trích xuất bảng customers và orders
customers = dfs['customers'].copy()
orders = dfs['orders'].copy()

# 2. Lọc khách hàng có age_group khác null
customers_with_age = customers[customers['age_group'].notnull()]

# 3. Đếm số đơn hàng của mỗi khách hàng từ bảng orders
# Kết quả trả về một Series với index là customer_id và giá trị là số đơn hàng
order_counts = orders['customer_id'].value_counts().reset_index()
order_counts.columns = ['customer_id', 'total_orders']

# 4. Join thông tin nhóm tuổi với số lượng đơn hàng
# Dùng left join để giữ lại tất cả khách hàng có nhóm tuổi (ngay cả khi họ chưa mua hàng - số đơn = 0)
age_order_merge = pd.merge(customers_with_age, order_counts, on='customer_id', how='left')

# 5. Xử lý các khách hàng không có đơn hàng nào (NaN chuyển thành 0)
age_order_merge['total_orders'] = age_order_merge['total_orders'].fillna(0)

# 6. Tính toán theo nhóm tuổi:
# - Tổng số đơn hàng trong nhóm
# - Số lượng khách hàng duy nhất trong nhóm
age_stats = age_order_merge.groupby('age_group').agg(
    total_orders_sum = ('total_orders', 'sum'),
    customer_count = ('customer_id', 'nunique')
)

# 7. Tính số đơn hàng trung bình trên mỗi khách hàng theo công thức đề bài
age_stats['avg_orders_per_customer'] = age_stats['total_orders_sum'] / age_stats['customer_count']

# 8. Tìm nhóm tuổi có giá trị cao nhất
best_age_group = age_stats['avg_orders_per_customer'].idxmax()
highest_avg = age_stats['avg_orders_per_customer'].max()

print("--- KẾT QUẢ Q6 ---")
print(age_stats.sort_values(by='avg_orders_per_customer', ascending=False))
print(f"\n=> Nhóm tuổi có số đơn hàng trung bình cao nhất là: {best_age_group} ({highest_avg:.2f} đơn/khách)")

--- KẾT QUẢ Q6 ---
           total_orders_sum  customer_count  avg_orders_per_customer
age_group                                                           
55+                 72760.0           13457                 5.406851
45-54              124138.0           23172                 5.357241
35-44              170368.0           31920                 5.337343
25-34              190622.0           36342                 5.245226
18-24               89057.0           17039                 5.226656

=> Nhóm tuổi có số đơn hàng trung bình cao nhất là: 55+ (5.41 đơn/khách)


In [11]:
# 1. Trích xuất các bảng cần thiết
orders = dfs['orders'].copy()
geography = dfs['geography'].copy()
payments = dfs['payments'].copy()

# 2. Kết nối bảng orders với geography để biết mỗi đơn hàng thuộc vùng nào
order_geo = pd.merge(orders, geography, on='zip', how='inner')

# 3. Kết nối kết quả trên với bảng payments để lấy giá trị doanh thu của mỗi đơn hàng
# Lưu ý: Một đơn hàng trong 'orders' tương ứng 1:1 với một bản ghi trong 'payments'
order_revenue = pd.merge(order_geo, payments, on='order_id', how='inner')

# 4. Tính tổng doanh thu (payment_value) theo từng Vùng (region)
region_sales = order_revenue.groupby('region')['payment_value'].sum().sort_values(ascending=False)

# 5. Tìm vùng có doanh thu cao nhất
best_region = region_sales.idxmax()
highest_revenue = region_sales.max()

print("--- KẾT QUẢ Q7 ---")
print("Tổng doanh thu theo từng vùng:")
print(region_sales)
print(f"\n=> Vùng tạo ra tổng doanh thu cao nhất là: {best_region}")

# Kiểm tra xem các vùng có xấp xỉ nhau không (để cân nhắc đáp án D)
revenue_diff_pct = (region_sales.max() - region_sales.min()) / region_sales.max()
print(f"Độ chênh lệch giữa vùng cao nhất và thấp nhất: {revenue_diff_pct:.2%}")

--- KẾT QUẢ Q7 ---
Tổng doanh thu theo từng vùng:
region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: payment_value, dtype: float64

=> Vùng tạo ra tổng doanh thu cao nhất là: East
Độ chênh lệch giữa vùng cao nhất và thấp nhất: 49.66%


In [12]:
# 1. Trích xuất bảng orders
orders = dfs['orders'].copy()

# 2. Lọc các đơn hàng có trạng thái là 'cancelled'
cancelled_orders = orders[orders['order_status'] == 'cancelled']

# 3. Đếm tần suất xuất hiện của các phương thức thanh toán trong nhóm này
payment_counts = cancelled_orders['payment_method'].value_counts()

# 4. Tìm phương thức thanh toán xuất hiện nhiều nhất
most_common_payment = payment_counts.idxmax()
max_count = payment_counts.max()

print("--- KẾT QUẢ Q8 ---")
print("Thống kê phương thức thanh toán của các đơn hàng bị hủy:")
print(payment_counts)
print(f"\n=> Phương thức thanh toán được sử dụng nhiều nhất trong các đơn hàng bị hủy là: {most_common_payment}")

--- KẾT QUẢ Q8 ---
Thống kê phương thức thanh toán của các đơn hàng bị hủy:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

=> Phương thức thanh toán được sử dụng nhiều nhất trong các đơn hàng bị hủy là: credit_card


In [13]:
# 1. Trích xuất các bảng cần thiết
order_items = dfs['order_items'].copy()
returns = dfs['returns'].copy()
products = dfs['products'].copy()

# 2. Tính số dòng đã bán theo từng kích thước
# Kết nối order_items với products để lấy cột 'size'
sold_with_size = pd.merge(order_items, products[['product_id', 'size']], on='product_id', how='inner')
sold_counts = sold_with_size['size'].value_counts()

# 3. Tính số dòng bị trả lại theo từng kích thước
# Kết nối returns với products để lấy cột 'size'
returned_with_size = pd.merge(returns, products[['product_id', 'size']], on='product_id', how='inner')
return_counts = returned_with_size['size'].value_counts()

# 4. Tính tỷ lệ trả hàng (Return Rate)
# Tỷ lệ = (Số bản ghi trong returns) / (Số dòng trong order_items)
return_rate = (return_counts / sold_counts).sort_values(ascending=False)

# 5. Lọc để chỉ lấy 4 kích thước yêu cầu (S, M, L, XL) và tìm Max
target_sizes = ['S', 'M', 'L', 'XL']
final_rates = return_rate.reindex(target_sizes)
highest_size = final_rates.idxmax()

print("--- KẾT QUẢ Q9 ---")
print("Tỷ lệ trả hàng theo từng kích thước:")
print(final_rates)
print(f"\n=> Kích thước có tỷ lệ trả hàng cao nhất là: {highest_size}")

--- KẾT QUẢ Q9 ---
Tỷ lệ trả hàng theo từng kích thước:
size
S     0.056515
M     0.055660
L     0.056250
XL    0.055200
Name: count, dtype: float64

=> Kích thước có tỷ lệ trả hàng cao nhất là: S


In [14]:
# 1. Trích xuất bảng payments
payments = dfs['payments'].copy()

# 2. Nhóm dữ liệu theo số kỳ trả góp (installments) 
# và tính giá trị trung bình (mean) của cột payment_value
installment_stats = payments.groupby('installments')['payment_value'].mean()

# 3. Tìm số kỳ trả góp có giá trị trung bình cao nhất
best_installment_plan = installment_stats.idxmax()
highest_avg_value = installment_stats.max()

print("--- KẾT QUẢ Q10 ---")
print("Giá trị thanh toán trung bình theo số kỳ trả góp:")
print(installment_stats.sort_values(ascending=False))

print(f"\n=> Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất là: {best_installment_plan} kỳ")
print(f"Giá trị trung bình: {highest_avg_value:.2f}")

--- KẾT QUẢ Q10 ---
Giá trị thanh toán trung bình theo số kỳ trả góp:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64

=> Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất là: 6 kỳ
Giá trị trung bình: 24446.65
